In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

Data processing journal

- first step is to clean and validate the data. Through this process we discard some data, as there are quotes with different prices but the same key.
- this is likely due to introduction of PM-settled monthly options, and the 
- in any case we have plenty of data remaining after simply discarding the invalid data, covering a long time period and interesting regimes regarding different historical events

# Data processing TODO:
- import data and create dataframes
- create documentation describing the dataframes
- export to csv files 



In [11]:


"""
Other assumptions about the raw data:
- The OptionKey is unique per day.
- The csvs are in directories named by year
"""


# daily csv files must contain these columns, or we will raise an error
REQUIRED_COLUMNS = {
    "OptionKey",
    "Symbol",
    "ExpirationDate",
    "AskPrice",
    "AskSize",
    "BidPrice",
    "BidSize",
    "LastPrice",
    "PutCall",
    "StrikePrice",
    "Volume",
    "OpenInterest",
    "UnderlyingPrice",
    "DataDate",
}

# we'll use some lovely Pythonic names for the columns
COLUMN_NAMES = {
    "OptionKey": "option_key",
    "Symbol": "symbol",
    "ExpirationDate": "expiration_date",
    "AskPrice": "ask_price",
    "AskSize": "ask_size",
    "BidPrice": "bid_price",
    "BidSize": "bid_size",
    "LastPrice": "last_price",
    "PutCall": "put_call",
    "StrikePrice": "strike_price",
    "Volume": "volume",
    "OpenInterest": "open_interest",
    "UnderlyingPrice": "underlying_price",
    "DataDate": "data_date",
}


# We expect data with these types. Note that here we are using pandas types, which
# support null values
READ_DTYPES = {
    "OptionKey": "string",
    "Symbol": "string",
    "AskPrice": "float64",
    "AskSize": "Int64",
    "BidPrice": "float64",
    "BidSize": "Int64",
    "LastPrice": "float64",
    "PutCall": "string",
    "StrikePrice": "float64",
    "Volume": "Int64",
    "OpenInterest": "Int64",
    "UnderlyingPrice": "float64",
}

# output only these columns
OUTPUT_COLUMNS = [
    "option_key",
    "symbol",
    "data_date",
    "expiration_date",
    "put_call",
    "strike_price",
    "underlying_price",
    "bid_price",
    "ask_price",
    "mid_price",
    "bid_ask_spread",
    "relative_bid_ask_spread",
    "bid_size",
    "ask_size",
    "last_price",
    "volume",
    "open_interest",
    "dte_calendar_days",
    "time_to_expiry_years_act36525",
    "moneyness_k_over_s",
    "log_moneyness_k_over_s",
    "quote_is_valid",
    "quote_is_two_sided",
    "is_otm",
    "source_file",
]

# the date format actually does change at one point so we need this
def parse_date_column(series):
    parsed = pd.to_datetime(
        series, format="%Y-%m-%d", errors="coerce"
    )

    unresolved = parsed.isna()

    parsed.loc[unresolved] = pd.to_datetime(
        series.loc[unresolved],
        format="%m/%d/%Y",
        errors="coerce",
    )

    return parsed

def read_and_prepare_daily_file(path: Path) -> pd.DataFrame:
    """
    Read, validate, and enrich one CSV options chain file.
    Returns a DataFrame with standardized column names, and some derived columns
    """

    raw_df = pd.read_csv(path, dtype=READ_DTYPES)

    missing = REQUIRED_COLUMNS.difference(raw_df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}")

    df = raw_df.loc[:, list(COLUMN_NAMES)].rename(columns=COLUMN_NAMES).copy()
    # df["review_flag"] = False  # new column to flag rows for review
    # df["data_date"] = pd.to_datetime(
    #     df["data_date"], format="%Y-%m-%d", errors="coerce"
    # )
    # df["expiration_date"] = pd.to_datetime(
    #     df["expiration_date"], format="%Y-%m-%d", errors="coerce"
    # )
    df["data_date"] = parse_date_column(df["data_date"])
    df["expiration_date"] = parse_date_column(df["expiration_date"])
    df["symbol"] = df["symbol"].str.strip().str.upper()
    df["put_call"] = df["put_call"].str.strip().str.lower()
    date_cols = ["data_date", "expiration_date"]
    bad_date_rows = df[date_cols].isna().any(axis=1)

    if bad_date_rows.any():
        print(
            f"Warning: {path.name}: discarding "
            f"{bad_date_rows.sum()} rows with invalid dates"
        )
        df = df.loc[~bad_date_rows].copy()

    if df.empty:
        raise ValueError(f"{path.name}: all rows have invalid dates")
    if df["data_date"].nunique() != 1:
        print(f"Warning: {path.name}: expected exactly one data date. Dropping {df['data_date'].duplicated(keep=False).sum()} rows.")
        df = df[~df["data_date"].duplicated(keep=False)]
    # if df["data_date"].iloc[0].normalize() != file_date(path):
    #     raise ValueError(f"{path.name}: filename date and DataDate do not match")
    if not df["put_call"].isin(["call", "put"]).all():
        print(f"Warning: {path.name}: PutCall must contain only call or put. Discarding {df[~df['put_call'].isin(['call', 'put'])].shape[0]} rows.")
        df = df[~df['put_call'].isin(['call', 'put'])]
    if (df["strike_price"] <= 0).any() or (df["underlying_price"] <= 0).any():
        print(f"Warning: {path.name}: strike and underlying prices must be positive. Discarding {df[(df['strike_price'] <= 0) | (df['underlying_price'] <= 0)].shape[0]} rows.")
        df = df[~((df["strike_price"] <= 0) | (df["underlying_price"] <= 0))]
    if (df["expiration_date"] < df["data_date"]).any():
        print(f"Warning: {path.name}: expiration before data date. Discarding {df[df['expiration_date'] < df['data_date']].shape[0]} rows.")
        df = df[~(df["expiration_date"] < df["data_date"])]
    if df.empty:
        print(f"Warning: {path.name}: all rows have been discarded due to validation errors")

    # A zero bid can occur for illiquid options.  Keep it, but flag the quote as
    # one-sided.  A non-positive ask or a crossed/negative quote is invalid.
    df["quote_is_valid"] = (
        (df["ask_price"] > 0)
        & (df["bid_price"] >= 0)
        & (df["ask_price"] >= df["bid_price"])
    ).astype("boolean")
    df["quote_is_two_sided"] = (
        df["quote_is_valid"] & (df["bid_price"] > 0)
    ).astype("boolean")

    midpoint = (df["bid_price"] + df["ask_price"]) / 2.0
    spread = df["ask_price"] - df["bid_price"]
    df["mid_price"] = midpoint.where(df["quote_is_valid"])
    df["bid_ask_spread"] = spread.where(df["quote_is_valid"])
    df["relative_bid_ask_spread"] = (
        df["bid_ask_spread"] / df["mid_price"]
    ).where(df["mid_price"] > 0)

    df["dte_calendar_days"] = (
        df["expiration_date"] - df["data_date"]
    ).dt.days.astype("Int32")
    df["time_to_expiry_years_act36525"] = df["dte_calendar_days"] / 365.25

    # Moneyness conventions differ, so the column name states the definition.
    df["moneyness_k_over_s"] = df["strike_price"] / df["underlying_price"]
    df["log_moneyness_k_over_s"] = np.log(df["moneyness_k_over_s"])
    df["is_otm"] = (
        ((df["put_call"] == "call") & (df["strike_price"] > df["underlying_price"]))
        | ((df["put_call"] == "put") & (df["strike_price"] < df["underlying_price"]))
    ).astype("boolean")

    df["source_file"] = path.name
    return df.loc[:, OUTPUT_COLUMNS]


def prepare_year(files: list[Path]) -> pd.DataFrame:
    """Combine all daily files for one year and perform annual checks."""
    annual = pd.concat(
        [read_and_prepare_daily_file(path) for path in files],
        ignore_index=True,
    )

    # note that this assumes options keys are unique per day
    # this is because in the raw data, the option key is a string that includes the data date
    duplicate_keys = annual["option_key"].duplicated(keep=False)
    invalid_data = pd.DataFrame()
    if duplicate_keys.any():
        invalid_data = annual.loc[duplicate_keys].copy()
        examples = invalid_data.sample(min(5, invalid_data.shape[0]))
        print(f"Warning: Duplicate option_key values found; examples: {examples}. Dropping all duplicates, in total {duplicate_keys.sum()} rows, which is {duplicate_keys.sum() / len(annual) * 100:.2f}% of the data.")
        annual = annual[~duplicate_keys]

    # first sort by data_date, then within data_date by expiration_date, etc.
    return annual.sort_values(
        ["data_date", "expiration_date", "put_call", "strike_price"],
        kind="stable",
        ignore_index=True,
    ), invalid_data

def discover_files(raw_dir: Path) -> dict[int, list[Path]]:
    """Find input files and group them by year. Assumes that the files are in a directory named by year."""
    files_by_year = {}
    for year_dir in raw_dir.iterdir():
        if not year_dir.is_dir():
            continue
        try:
            year = int(year_dir.name)
        except ValueError:
            print(f"Warning: skipping non-integer directory {year_dir}")
            continue
        files_by_year[year] = list(year_dir.glob("*.csv"))
    return files_by_year


def write_parquet(df: pd.DataFrame, path: Path, overwrite: bool) -> None:
    """Write one compressed annual Parquet file."""
    if path.exists() and not overwrite:
        print(f"Warning: {path} already exists; skipping write")
        return
    path.parent.mkdir(parents=True, exist_ok=True) # creates the directory, as well as the parent directories, and does not raise an error if they already exist

    df.to_parquet(
        path,
        engine="pyarrow",
        compression="zstd",
        index=False,
    )

input_files = discover_files(Path("raw/SPX"))
input_years = sorted(input_files)
for year in input_years:
    annual, invalid_data = prepare_year(input_files[year])
    valid = int(annual["quote_is_valid"].sum())
    two_sided = int(annual["quote_is_two_sided"].sum())
    message = (
        f"{year}: {len(input_files[year])} files, {len(annual):,} rows, "
        f"{valid:,} valid quotes, {two_sided:,} two-sided quotes"
    )
    message_invalid = (
        f"{year}: {len(invalid_data)} rows with duplicate option_key values"
    )
    output_path = Path(f"processed/spx_options_{year}.parquet")
    write_parquet(annual, output_path, overwrite=True)
    write_parquet(invalid_data, Path(f"processed/spx_options_{year}_invalid.parquet"), overwrite=True)
    print(message + f" -> {output_path})")
    print(message_invalid + f" -> {output_path.with_name(output_path.stem + '_invalid.parquet')}")



2005: 252 files, 150,453 rows, 150,108 valid quotes, 138,384 two-sided quotes -> processed/spx_options_2005.parquet)
2005: 0 rows with duplicate option_key values -> processed/spx_options_2005_invalid.parquet
2006: 251 files, 170,219 rows, 170,190 valid quotes, 158,755 two-sided quotes -> processed/spx_options_2006.parquet)
2006: 0 rows with duplicate option_key values -> processed/spx_options_2006_invalid.parquet
2007: 251 files, 222,678 rows, 222,674 valid quotes, 209,594 two-sided quotes -> processed/spx_options_2007.parquet)
2007: 0 rows with duplicate option_key values -> processed/spx_options_2007_invalid.parquet
292816  SPX2008-12-20p800.002008-01-03    SPX 2008-01-03      2008-12-20   
292817  SPX2008-12-20p800.002008-01-03    SPX 2008-01-03      2008-12-20   
292814  SPX2008-12-20c800.002008-01-03    SPX 2008-01-03      2008-12-20   
292815  SPX2008-12-20c800.002008-01-03    SPX 2008-01-03      2008-12-20   

       put_call  strike_price  underlying_price  bid_price  ask_pric

In [ ]:
good_years = [2005, 2006, 2007, 2008, 2009, 2010, 2014, 2015, 2016, 2018, 2021, 2022, 2023, 2024]

In [4]:
df = pd.read_parquet("processed/spx_options_2005.parquet")
df.head()
# df.describe()

# df.loc[df["quote_is_valid"].isna()].head()

,option_key,symbol,data_date,expiration_date,put_call,strike_price,underlying_price,bid_price,ask_price,mid_price,...,volume,open_interest,dte_calendar_days,time_to_expiry_years_act36525,moneyness_k_over_s,log_moneyness_k_over_s,quote_is_valid,quote_is_two_sided,is_otm,source_file
0,SPX2005-01-22c500.002005-01-03,SPX,2005-01-03,2005-01-22,call,500.0,1202.08,703.7,705.7,704.7,...,0,950,19,0.052019,0.415946,-0.877201,True,True,False,20050103_OData_SPX.csv
1,SPX2005-01-22c600.002005-01-03,SPX,2005-01-03,2005-01-22,call,600.0,1202.08,603.8,605.8,604.8,...,0,0,19,0.052019,0.499135,-0.694879,True,True,False,20050103_OData_SPX.csv
2,SPX2005-01-22c650.002005-01-03,SPX,2005-01-03,2005-01-22,call,650.0,1202.08,553.8,555.8,554.8,...,0,0,19,0.052019,0.540729,-0.614836,True,True,False,20050103_OData_SPX.csv
3,SPX2005-01-22c700.002005-01-03,SPX,2005-01-03,2005-01-22,call,700.0,1202.08,503.9,505.9,504.9,...,0,100,19,0.052019,0.582324,-0.540728,True,True,False,20050103_OData_SPX.csv
4,SPX2005-01-22c750.002005-01-03,SPX,2005-01-03,2005-01-22,call,750.0,1202.08,454.0,456.0,455.0,...,0,0,19,0.052019,0.623919,-0.471735,True,True,False,20050103_OData_SPX.csv
